# SignBridge: fine-tune Google's gesture model on our ISL signs

**Transfer learning.** We start from Google's **pre-trained MediaPipe Gesture Recognizer**: a hand-landmark model plus a gesture
embedding model trained by Google on a large hand dataset. Only the final classification layer is retrained, on photos of
**our own right-hand ISL signs**. That's why a few hundred photos and about 10 minutes on a free Colab runtime are enough.

**Input:** the zip from the SignBridge app (Train tab → Data → *Download photo dataset (.zip)*). It has one folder per sign,
and `none` for the resting hand, which this tool requires.

**Output:** `signbridge_gestures.task`. Put it in `app/models/`, and the app's *SignBridge fine-tuned ISL model* option switches on.

Run the cells in order (Runtime → Run all works too).

## 1. Install MediaPipe Model Maker

In [ ]:
!pip install -q mediapipe-model-maker

If the install fails because of the Python version, choose **Runtime → Change runtime type** and pick an older runtime
version, or run `!pip install -q mediapipe-model-maker==0.2.1.4` instead.

## 2. Upload the photo zip from the app

In [ ]:
import os, zipfile, glob
from google.colab import files

uploaded = files.upload()                 # pick signbridge-photos-YYYY-MM-DD.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall('data')
DATASET = glob.glob('data/*/')[0] if not os.path.isdir('data/none') else 'data'
print('Dataset folder:', DATASET)
for label in sorted(os.listdir(DATASET)):
    print(f'{label:15s}', len(os.listdir(os.path.join(DATASET, label))), 'photos')

## 3. Load the photos (finds the hand in each one)

In [ ]:
from mediapipe_model_maker import gesture_recognizer

data = gesture_recognizer.Dataset.from_folder(
    dirname=DATASET,
    hparams=gesture_recognizer.HandDataPreprocessingParams())
train_data, rest = data.split(0.8)
validation_data, test_data = rest.split(0.5)
print('train', len(train_data), '· validation', len(validation_data), '· test', len(test_data))

Photos where no hand is found are skipped, so these counts can be lower than the number of photos.

## 4. Fine-tune

In [ ]:
hparams = gesture_recognizer.HParams(export_dir='exported_model', epochs=30, batch_size=8, learning_rate=0.001)
options = gesture_recognizer.GestureRecognizerOptions(
    model_options=gesture_recognizer.ModelOptions(dropout_rate=0.1),
    hparams=hparams)
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data, validation_data=validation_data, options=options)

## 5. Test on photos the model never saw

In [ ]:
loss, accuracy = model.evaluate(test_data, batch_size=1)
print(f'Test accuracy: {accuracy * 100:.1f}%   (loss {loss:.3f})')

Report this number together with the dataset size, how many people recorded, and the lighting.
Photos from the same recording can land in both train and test, so for a stricter number record a separate test session.

## 6. Export and download the model

In [ ]:
model.export_model(model_name='signbridge_gestures.task')
files.download('exported_model/signbridge_gestures.task')

Copy the downloaded file to `app/models/signbridge_gestures.task` in the repo, commit it, and reload the app.
In the Talk tab choose **Model → SignBridge fine-tuned ISL model**.